# 01 EDA: seasonality, lags, durations
Run from `notebooks/` (kernel cwd) or project root.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'pyproject.toml').exists() else ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks'))

import pandas as pd

from src.data_loader import load_raw

grid = pd.read_csv(ROOT / 'data/processed/grid.csv', parse_dates=['timestamp'], index_col='timestamp')
grid.index = pd.to_datetime(grid.index, utc=True)
print(grid.shape)
print('Kyiv alert rate:', round(grid['Kyiv'].mean(), 4))
print('hours:', grid.index.min(), '->', grid.index.max())

(40495, 5)
Kyiv alert rate: 0.1185
hours: 2022-02-25 16:00:00+00:00 -> 2026-10-09 22:00:00+00:00


In [2]:
local = grid.index.tz_convert('Europe/Kyiv')
print('prevalence by Kyiv-local hour:')
print(grid['Kyiv'].groupby(local.hour).mean().round(3).to_string())
print('prevalence by weekday:')
print(grid['Kyiv'].groupby(local.dayofweek).mean().round(3).to_string())

prevalence by Kyiv-local hour:
timestamp
0     0.145
1     0.154
2     0.150
3     0.160
4     0.154
5     0.142
6     0.119
7     0.108
8     0.100
9     0.107
10    0.112
11    0.122
12    0.128
13    0.126
14    0.112
15    0.104
16    0.088
17    0.089
18    0.093
19    0.100
20    0.094
21    0.102
22    0.105
23    0.129
prevalence by weekday:
timestamp
0    0.117
1    0.119
2    0.117
3    0.132
4    0.122
5    0.122
6    0.102


In [3]:
print('lag_1==1 -> P(Y=1 next 3h, Y=max(A[t..t+2])):')
a = grid['Kyiv']
y = ((a == 1) | (a.shift(-1) == 1) | (a.shift(-2) == 1)).astype(int)
print(y[a.shift(1) == 1].mean().round(3), ' vs onset:', y[a.shift(1) == 0].mean().round(3))


lag_1==1 -> P(Y=1 next 3h, Y=max(A[t..t+2])):
0.625  vs onset: 0.17


In [4]:
raw = load_raw(ROOT / 'data/raw/alerts.csv')
dur = (raw['end_time'] - raw['start_time']).dt.total_seconds() / 60
print(dur.describe()[['count', 'mean', '50%', 'max']].to_string())
print('share < 60min:', round((dur < 60).mean(), 3))

2026-10-11 22:34:07.589 | WARNING  | src.data_loader:normalize_regions:175 - unknown region names kept as-is: ['cherkaska oblast', 'chernivetska oblast', 'dnipropetrovska oblast', 'donetska oblast', 'ivano-frankivska oblast', 'kharkivska oblast', 'khersonska oblast', 'khmelnytska oblast', 'kirovohradska oblast', 'luhanska oblast', 'lvivska oblast', 'mykolaivska oblast', 'odeska oblast', 'rivnenska oblast', 'ternopilska oblast', 'vinnytska oblast', 'volynska oblast', 'zakarpatska oblast', 'zaporizka oblast', 'zhytomyrska oblast']


count    115787.0000
mean         47.8342
50%          30.0000
max       10695.9000
share < 60min: 0.778
